In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path


In [2]:
folder_path = "individual_stock_data" #input
output_folder = "processed_stock_data" #output

In [3]:
os.makedirs("processed_stock_data", exist_ok=True)#no folder big crash

In [9]:
def calculate_various_parameters(df, timeframe):
    roll = df["Returns"].rolling(timeframe)
    #calculating the indicators needed
    df[f"Mean_{timeframe}"] = roll.mean()
    df[f"Std_{timeframe}"] = roll.std()
    df[f'Z_Score_{timeframe}'] = (df['Returns'] - df[f'Mean_{timeframe}']) / df[f'Std_{timeframe}']
    df[f'Skew_{timeframe}']    = roll.skew()
    #E[ ((x - mean) / std)⁴ ] - 3 for kurtosis
    df[f'Kurt_{timeframe}']    = roll.kurt()
    #Calculation of basic things ends here
    #returns all the values foe us
    return (df[f'Mean_{timeframe}'], df[f'Std_{timeframe}'],
            df[f'Z_Score_{timeframe}'], df[f'Skew_{timeframe}'], df[f'Kurt_{timeframe}'])
#Kinda copied this from the already availible code
#sqrt( mean( log(High/Low)² ) / (4 × ln(2)) ) for rolling parkinson volatility

def rolling_parkinson_vol(df, window):
    hl_log_sq = np.log(df['High'] / df['Low']) ** 2
    return np.sqrt(hl_log_sq.rolling(window).mean() / (4 * np.log(2)))
#sqrt( mean( 0.5 * log(High/Low)² - (2 * ln(2) - 1) * log(Close/Open)² ) ) for rolling garman-klass volatility

def rolling_garman_klass_vol(df, timeframe):
    hl = np.log(df['High'] / df['Low']) ** 2
    co = np.log(df['Close'] / df['Open']) ** 2
    return np.sqrt((0.5 * hl - (2 * np.log(2) - 1) * co).rolling(timeframe).mean())
#sqrt( mean( log(High/Low)² + log(Open/Close)² + log(Close/Open_prev)² ) / 3 ) for rolling yang-zhang volatility
def rolling_yang_zhang_vol(df, timeframe):
    log_ho = np.log(df['High']  / df['Open'])
    log_lo = np.log(df['Low']   / df['Open'])
    log_co = np.log(df['Close'] / df['Open'])
    log_oc = np.log(df['Open']  / df['Close'].shift(1))
    rs     = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)
    sigma_oc = log_oc.rolling(timeframe).var()
    sigma_co = log_co.rolling(timeframe).var()
    sigma_rs = rs.rolling(timeframe).mean()
    k = 0.34 / (1.34 + (timeframe + 1) / (timeframe - 1))
    return np.sqrt(sigma_oc + k * sigma_co + (1 - k) * sigma_rs)
def rolling_linear_fit(series, timeframe):
    slopes = np.full(len(series), np.nan)
    mses   = np.full(len(series), np.nan)
    x      = np.arange(timeframe)
    for i in range(timeframe - 1, len(series)):
        y         = series.iloc[i - timeframe + 1 : i + 1].values
        x_mean, y_mean = x.mean(), y.mean()
        num       = np.sum((x - x_mean) * (y - y_mean))
        den       = np.sum((x - x_mean) ** 2)
        slope     = num / den
        intercept = y_mean - slope * x_mean
        y_hat     = slope * x + intercept
        slopes[i] = slope
        mses[i]   = np.mean((y - y_hat) ** 2)
    return slopes, mses

for filename in os.listdir(folder_path):
    if not filename.endswith(".csv"):
        continue

    df = pd.read_csv(os.path.join(folder_path, filename))

    # Price ratio features
    df['OH_ratio']     = (df['Open']  - df['High'])  / df['Open']
    df['OL_ratio']     = (df['Open']  - df['Low'])   / df['Open']
    df['CH_ratio']     = (df['Close'] - df['High'])  / df['Close']
    df['CL_ratio']     = (df['Close'] - df['Low'])   / df['Close']
    df['Price_Change'] = (df['Close'] - df['Open'])  / df['Open']
    df['Returns']      = df['Close'].pct_change()
    df['Day_Range']    = (df['High']  - df['Low'])   / df['Low']

    # Statistical + volatility + regression features at 5 windows
    for w in [5, 20, 50, 100, 200]:
        (df[f'Mean_{w}'], df[f'Std_{w}'], df[f'Z_Score_{w}'],#for some dmn reason Z score should have capitalised S
         #error cause you got the spelling of calculate wrong in the function
         #should check the spellings of functions ffs
        df[f'Skew_{w}'], df[f'Kurt_{w}']) = calculate_various_parameters(df, w)

        df[f'Parkinson_Vol_{w}'] = rolling_parkinson_vol(df, w)
        df[f'GK_Vol_{w}']        = rolling_garman_klass_vol(df, w)
        df[f'YZ_Vol_{w}']        = rolling_yang_zhang_vol(df, w)
        slope, mse = rolling_linear_fit(df['Returns'], w)
        df[f'Returns_Slope_{w}'] = slope
        df[f'Returns_MSE_{w}']   = mse

    df.to_csv(os.path.join(output_folder, filename), index=False)
    print(f"Processed: {filename}")

print("Done!")


Processed: 63MOONS.csv
Processed: AARTIDRUGS.csv
Processed: ADVENZYMES.csv
Processed: AGIIL.csv
Processed: AJMERA.csv
Processed: ALEMBICLTD.csv
Processed: ANUP.csv
Processed: ARSSBL.csv
Processed: ARVSMART.csv
Processed: ASHIANA.csv
Processed: ASHOKA.csv
Processed: AUTOAXLES.csv
Processed: AVANTEL.csv
Processed: AWFIS.csv
Processed: BAJAJCON.csv
Processed: BALAMINES.csv
Processed: BALMLAWRIE.csv
Processed: BBL.csv
Processed: BHAGCHEM.csv
Processed: BHARATRAS.csv
Processed: BOMDYEING.csv
Processed: BOROLTD.csv
Processed: BOSCH-HCIL.csv
Processed: BRIGHOTEL.csv
Processed: CAMLINFINE.csv
Processed: CARRARO.csv
Processed: CENTUM.csv
Processed: CHEMPLASTS.csv
Processed: CYIENTDLM.csv
Processed: DATAMATICS.csv
Processed: DCAL.csv
Processed: DDEVPLSTIK.csv
Processed: DIAMONDYD.csv
Processed: DPABHUSHAN.csv
Processed: DREDGECORP.csv
Processed: E2E.csv
Processed: EASEMYTRIP.csv
Processed: EBGNG.csv
Processed: EFCIL.csv
Processed: EIEL.csv
Processed: ELLEN.csv
Processed: EMIL.csv
Processed: EPAC